In [ ]:
import pandas as pd
import random
import logging
import requests
from faker import Faker
from sqlalchemy import create_engine, text

fake = Faker("en_US")

# ------------------------------------------------
# LOGGING CONFIG
# ------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)




# ================================================
# STEP 1: DATA EXTRACTION FROM APIs
# ================================================

def extract_dummyjson_products() -> pd.DataFrame:
    """Extract product data from DummyJSON API."""
    logger.info("Extracting data from DummyJSON API...")
    url = "https://dummyjson.com/products?limit=200"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    records = []
    for p in data["products"]:
        records.append({
            "product_name": p["title"],
            "brand":        p.get("brand", "Unknown"),
            "category":     p["category"],
            "price":        p["price"],
            "source":       "dummyjson"
        })

    df = pd.DataFrame(records)
    logger.info(f"DummyJSON: {len(df)} products extracted")
    return df


def extract_openfoodfacts_products() -> pd.DataFrame:
    """Extract product data from Open Food Facts API."""
    logger.info("Extracting data from Open Food Facts API...")
    url = "https://world.openfoodfacts.org/cgi/search.pl?action=process&json=true&page_size=100"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    records = []
    for p in data["products"]:
        records.append({
            "product_name": p.get("product_name", "Unknown"),
            "brand":        p.get("brands", "Unknown"),
            "category":     p.get("categories", "grocery"),
            "price":        None,
            "source":       "openfoodfacts"
        })

    df = pd.DataFrame(records)
    logger.info(f"Open Food Facts: {len(df)} products extracted")
    return df


def extract_all_data() -> pd.DataFrame:
    """Extract and combine data from all API sources."""
    dummy_df = extract_dummyjson_products()
    food_df  = extract_openfoodfacts_products()

    combined_df = pd.concat([dummy_df, food_df], ignore_index=True)
    logger.info(f"Total combined products: {len(combined_df)}")
    return combined_df


# ================================================
# STEP 2: DATA CLEANING
# ================================================

def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize and clean the raw combined product DataFrame."""
    df = df.copy()

    # Normalize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w]+", "_", regex=True)
        .str.strip("_")
    )

    # Rename discount column if present
    if "discount" in df.columns:
        df = df.rename(columns={"discount": "discount_percent"})

    required_cols = [
        "product_id", "sku", "product_name", "brand", "category", "subcategory",
        "description", "original_price", "discount_percent", "sale_price",
        "rating", "review_count", "weight_kg", "color", "size",
        "is_featured", "is_new_arrival", "is_bestseller", "date_added",
        "tags", "image_url", "product_url"
    ]
    df = df[required_cols]

    # Convert boolean columns
    bool_cols = ["is_featured", "is_new_arrival", "is_bestseller"]
    for col in bool_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.lower()
            .map({"yes": True, "no": False, "true": True, "false": False})
        )

    logger.info(f"Products after cleaning: {len(df)}")
    return df


# ================================================
# STEP 3: MASTER TABLE GENERATION
# ================================================

def create_categories(df: pd.DataFrame) -> pd.DataFrame:
    cat = df[["category"]].drop_duplicates().reset_index(drop=True)
    cat["category_id"] = range(1, len(cat) + 1)
    cat = cat.rename(columns={"category": "category_name"})
    logger.info(f"Categories created: {len(cat)}")
    return cat[["category_id", "category_name"]]


def create_subcategories(df: pd.DataFrame, categories: pd.DataFrame) -> pd.DataFrame:
    sub = df[["subcategory", "category"]].drop_duplicates()
    sub = sub.merge(categories, left_on="category", right_on="category_name")
    sub["subcategory_id"] = range(1, len(sub) + 1)
    sub = sub.rename(columns={"subcategory": "subcategory_name"})
    logger.info(f"Subcategories created: {len(sub)}")
    return sub[["subcategory_id", "subcategory_name", "category_id"]]


def create_brands(df: pd.DataFrame) -> pd.DataFrame:
    brands = df[["brand"]].drop_duplicates().reset_index(drop=True)
    brands["brand_id"] = range(1, len(brands) + 1)
    brands = brands.rename(columns={"brand": "brand_name"})
    logger.info(f"Brands created: {len(brands)}")
    return brands[["brand_id", "brand_name"]]


def create_products_master(
    df: pd.DataFrame,
    brands: pd.DataFrame,
    subcategories: pd.DataFrame
) -> pd.DataFrame:
    p = df.merge(brands, left_on="brand", right_on="brand_name")
    p = p.merge(
        subcategories[["subcategory_name", "subcategory_id"]],
        left_on="subcategory",
        right_on="subcategory_name"
    )
    p = p.drop_duplicates(subset=["product_id"])

    cols = [
        "product_id", "sku", "product_name",
        "brand_id", "subcategory_id",
        "description", "original_price",
        "discount_percent", "sale_price",
        "weight_kg", "color", "size",
        "is_featured", "is_new_arrival",
        "is_bestseller", "date_added",
        "tags", "image_url", "product_url"
    ]
    logger.info(f"Products master created: {len(p)}")
    return p[cols]


def create_suppliers() -> pd.DataFrame:
    names = [
        "Global Supply Inc",
        "Prime Retail Distributors",
        "USA Wholesale Hub",
        "North America Traders",
        "SuperGoods Distribution",
        "Retail Supply Chain Co",
        "American Product Supply",
        "FastTrack Distributors",
        "National Retail Supply",
        "Premium Wholesale Group"
    ]
    return pd.DataFrame({
        "supplier_id":   range(1, len(names) + 1),
        "supplier_name": names
    })


def create_product_supplier_map(
    products: pd.DataFrame,
    suppliers: pd.DataFrame
) -> pd.DataFrame:
    rows = []
    for pid in products["product_id"]:
        count        = random.randint(1, 4)
        supplier_ids = random.sample(list(suppliers["supplier_id"]), count)
        for sid in supplier_ids:
            rows.append({
                "product_id":        pid,
                "supplier_id":       sid,
                "supply_price":      round(random.uniform(5, 500), 2),
                "lead_time_days":    random.randint(2, 15),
                "minimum_order_qty": random.randint(5, 100)
            })
    logger.info(f"Product-supplier mappings created: {len(rows)}")
    return pd.DataFrame(rows)


def create_customers(start_date: str, end_date: str, n: int = 500) -> pd.DataFrame:
    """Generate fake customers with created_date between start_date and end_date.

    Args:
        start_date: Lower bound for created_date, e.g. "2022-01-01"
        end_date:   Upper bound for created_date, e.g. "2024-12-31"
        n:          Number of customer records to generate (default 500)
    """
    rows = []
    for _ in range(n):
        rows.append({
            "first_name":       fake.first_name(),
            "last_name":        fake.last_name(),
            "email":            fake.email(),
            "phone":            fake.phone_number()[:10],
            "city":             fake.city(),
            "state":            fake.state(),
            "country":          "USA",
            "postal_code":      fake.zipcode(),
            "customer_segment": random.choice(["Regular", "Premium", "VIP"]),
            "created_date":     fake.date_between(start_date=start_date, end_date=end_date)
        })
    logger.info(f"Customers created: {n} (date range: {start_date} → {end_date})")
    return pd.DataFrame(rows)


# ================================================
# STEP 4: DATABASE OPERATIONS
# ================================================

def truncate_tables(engine):
    tables = [
        "product_supplier_map",
        "suppliers",
        "products_master",
        "brands",
        "product_subcategories",
        "product_categories",
        "customers"
    ]
    with engine.begin() as conn:
        for t in tables:
            logger.info(f"Truncating {t}")
            conn.execute(text(f"TRUNCATE TABLE {t} CASCADE"))


def load_to_db(df: pd.DataFrame, table: str, engine):
    logger.info(f"Loading → {table} ({len(df)} rows)")
    df = df.reset_index(drop=True)
    df = df.loc[:, ~df.columns.duplicated()]
    df.to_sql(
        table,
        engine,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=500
    )


# ================================================
# MAIN
# ================================================

def main(
    host: str,
    port: int,
    db_name: str,
    username: str,
    password: str,
    start_date: str,
    end_date: str,
    n_customers: int = 500
):
    """Entry point for the retail ETL pipeline.

    Args:
        host:         Database host,        e.g. "localhost"
        port:         Database port,        e.g. 5432
        db_name:      Database name,        e.g. "Retail"
        username:     Database username,    e.g. "postgres"
        password:     Database password,    e.g. "secret"
        start_date:   Customer date lower bound, e.g. "2022-01-01"
        end_date:     Customer date upper bound, e.g. "2024-12-31"
        n_customers:  Number of fake customers to generate (default 500)
    """
    logger.info("=== Pipeline Started ===")
    logger.info(f"DB: {username}@{host}:{port}/{db_name}")
    logger.info(f"Customer date range: {start_date} → {end_date}")

    # Build DB engine from credentials
    db_uri = f"postgresql://{username}:{password}@{host}:{port}/{db_name}"
    engine = create_engine(db_uri)

    # 1. Extract raw data from APIs
    raw_df = extract_all_data()

    # 2. Clean the combined DataFrame
    clean_df = clean_products(raw_df)

    # 3. Build all dimension & master tables (in-memory DataFrames only)
    categories    = create_categories(clean_df)
    subcategories = create_subcategories(clean_df, categories)
    brands        = create_brands(clean_df)
    products      = create_products_master(clean_df, brands, subcategories)
    suppliers     = create_suppliers()
    supplier_map  = create_product_supplier_map(products, suppliers)
    customers     = create_customers(start_date, end_date, n=n_customers)

    # 4. Load to database
    truncate_tables(engine)
    load_to_db(categories,    "product_categories",   engine)
    load_to_db(subcategories, "product_subcategories", engine)
    load_to_db(brands,        "brands",               engine)
    load_to_db(products,      "products_master",      engine)
    load_to_db(suppliers,     "suppliers",            engine)
    load_to_db(supplier_map,  "product_supplier_map", engine)
    load_to_db(customers,     "customers",            engine)

    logger.info("=== Pipeline Completed Successfully ===")


: 

In [ ]:

if __name__ == "__main__":
    main(
        host       = "localhost",
        port       = 5432,
        db_name    = "Re***",
        username   = "postgres",
        password   = "****",
        start_date = "2022-01-01",
        end_date   = "2024-12-31",
        n_customers= 500
    )